# 03.5.1 FastF1 Data Reproduction

**Purpose**: Reproduce Kaggle-equivalent data from FastF1 sources, starting with 2024 validation, then producing 2025 data.

**Scope**: This notebook maps all required columns from FastF1 data to match the structure of `master_races_clean.csv`, calculates standings, and handles sprint results in the same row structure.

**Output**: 
- Validated 2024 reproduction (compared against master)
- 2025 production data (to append to master for rolling calculations)


## Setup: Imports, Paths, and Helper Functions


In [76]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# Get project root (works whether running from notebooks/ or F1/ folder)
PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

# Paths
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
KAGGLE_DIR = RAW_DATA_DIR / 'kaggle'
FASTF1_DIR = RAW_DATA_DIR / 'fastf1_2018plus'

# Create output directories if needed
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project Structure:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  KAGGLE_DIR: {KAGGLE_DIR}")
print(f"  FASTF1_DIR: {FASTF1_DIR}")
print(f"  PROCESSED_DATA_DIR: {PROCESSED_DATA_DIR}")
print()

# Verify directories exist
assert KAGGLE_DIR.exists(), f"Kaggle data directory not found: {KAGGLE_DIR}"
assert FASTF1_DIR.exists(), f"FastF1 data directory not found: {FASTF1_DIR}"
print("✓ All directories exist")


Project Structure:
  PROJECT_ROOT: C:\Users\erikv\Downloads\F1
  KAGGLE_DIR: C:\Users\erikv\Downloads\F1\data\raw\kaggle
  FASTF1_DIR: C:\Users\erikv\Downloads\F1\data\raw\fastf1_2018plus
  PROCESSED_DATA_DIR: C:\Users\erikv\Downloads\F1\data\processed

✓ All directories exist


In [77]:
# Load lookup tables
print("Loading lookup tables...")

# Circuits lookup
circuits_df = pd.read_csv(KAGGLE_DIR / 'circuits.csv', low_memory=False)
print(f"  ✓ Loaded circuits.csv: {len(circuits_df)} circuits")

# Drivers lookup
drivers_df = pd.read_csv(KAGGLE_DIR / 'drivers.csv', low_memory=False)
print(f"  ✓ Loaded drivers.csv: {len(drivers_df)} drivers")

# Status lookup (for mapping FastF1 Status text to statusId)
status_df = pd.read_csv(KAGGLE_DIR / 'status.csv', low_memory=False)
print(f"  ✓ Loaded status.csv: {len(status_df)} status types")

# Create status lookup dictionary: Status text -> statusId
status_lookup = dict(zip(status_df['status'].str.strip(), status_df['statusId']))
print(f"  ✓ Created status lookup dictionary: {len(status_lookup)} mappings")



Loading lookup tables...
  ✓ Loaded circuits.csv: 77 circuits
  ✓ Loaded drivers.csv: 861 drivers
  ✓ Loaded status.csv: 139 status types
  ✓ Created status lookup dictionary: 139 mappings


In [78]:
# Helper Functions

def ensure_driver_ids(drivers_df, fastf1_results):
    """
    Ensure every FastF1 driver code (Session == 'R') has a driverId.
    - If duplicate codes exist in drivers.csv, keep the max driverId.
    - If a code is missing, create a new row with driverId = max+1.
    Returns updated drivers_df and driver_code_to_id.
    """
    df = drivers_df.copy()

    df['code_norm'] = df['code'].astype(str).str.strip().str.upper()
    df['driverId_num'] = pd.to_numeric(df['driverId'], errors='coerce')

    # Dedup by code -> keep max driverId
    dedup = (
        df.dropna(subset=['code_norm', 'driverId_num'])
        .sort_values('driverId_num')
        .groupby('code_norm', as_index=False)
        .tail(1)
    )

    driver_code_to_id = dict(zip(dedup['code_norm'], dedup['driverId_num'].astype(int)))

    # FastF1 codes (race session only)
    fastf1_codes = set(
        fastf1_results[fastf1_results['Session'] == 'R']['Abbreviation']
        .dropna().astype(str).str.strip().str.upper().tolist()
    )

    missing_codes = sorted(list(fastf1_codes - set(driver_code_to_id.keys())))
    if missing_codes:
        next_id = int(dedup['driverId_num'].max()) + 1 if len(dedup) > 0 else 1
        new_rows = []
        for code in missing_codes:
            new_rows.append({
                'driverId': next_id,
                'driverRef': code.lower(),
                'number': pd.NA,
                'code': code,
                'forename': pd.NA,
                'surname': pd.NA,
                'dob': pd.NA,
                'nationality': pd.NA,
                'url': pd.NA
            })
            next_id += 1

        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

        # Rebuild map after adding new rows
        df['code_norm'] = df['code'].astype(str).str.strip().str.upper()
        df['driverId_num'] = pd.to_numeric(df['driverId'], errors='coerce')
        dedup = (
            df.dropna(subset=['code_norm', 'driverId_num'])
            .sort_values('driverId_num')
            .groupby('code_norm', as_index=False)
            .tail(1)
        )
        driver_code_to_id = dict(zip(dedup['code_norm'], dedup['driverId_num'].astype(int)))

    return df, driver_code_to_id

def map_event_schedule_info(race_df, event_schedule_df):
    """
    Map round + race date from FastF1 ALL_EVENT_SCHEDULE.csv
    using (Year, EventName) == (Year, Event).
    """
    if len(race_df) == 0:
        return race_df

    race_df = race_df.copy()

    # Build lookup: (Year, EventName) -> round/date
    schedule_lookup = {}
    for _, row in event_schedule_df.iterrows():
        key = (int(row['Year']), str(row['EventName']).strip())
        schedule_lookup[key] = {
            'round': int(row['RoundNumber']) if pd.notna(row['RoundNumber']) else pd.NA,
            # Prefer EventDate (race day). You can swap to Session5Date if you want exact race start time.
            'date': pd.to_datetime(row['EventDate'], errors='coerce')
        }

    def get_sched(row):
        return schedule_lookup.get((int(row['Year']), str(row['Event']).strip()), {})

    info = race_df.apply(get_sched, axis=1)
    race_df['round'] = [x.get('round', pd.NA) for x in info]
    race_df['date'] = [x.get('date', pd.NaT) for x in info]

    return race_df

def parse_time_to_ms(val):
    """Parse time string to pandas Timedelta (handles FastF1 + master formats)."""
    if pd.isna(val):
        return pd.NaT
    s = str(val).strip()
    if s in ["", "\\N", "NaT", "None", "nan"]:
        return pd.NaT

    s = s.replace("\u202f", "").replace("\xa0", "").strip()
    if s.lower().endswith("lap"):
        return pd.NaT

    # Strip "0 days" if present
    if "days" in s:
        s = s.split("days", 1)[1].strip()

    # Normalize to HH:MM:SS.mmm for pd.to_timedelta
    if s.startswith("+"):
        s = s[1:].strip()
    if re.match(r"^\d+:\d{2}\.\d+$", s) or re.match(r"^\d+:\d{2}\.\d{3,6}$", s):
        s = "00:" + s
    elif re.match(r"^\d+:\d{2}:\d{2}$", s):
        s = s + ".000"
    elif re.match(r"^\d+\.\d+$", s):
        s = "00:00:" + s

    try:
        return pd.to_timedelta(s).round("1ms")
    except Exception:
        return pd.NaT


def time_to_milliseconds(time_val):
    """Convert time value to milliseconds."""
    td = parse_time_to_ms(time_val)
    if pd.isna(td):
        return pd.NA
    return int(round(td.total_seconds() * 1000))


def format_timedelta_to_time_str(td):
    """Format Timedelta to H:MM:SS.mmm or M:SS.mmm."""
    if pd.isna(td):
        return pd.NA
    total_seconds = td.total_seconds()
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)
    seconds = total_seconds % 60
    if hours > 0:
        return f"{hours}:{minutes:02d}:{seconds:06.3f}"
    return f"{minutes}:{seconds:06.3f}"


def normalize_quali_time(val):
    """Convert FastF1 timedelta string to MM:SS.mmm (or NA)."""
    td = parse_time_to_ms(val)
    if pd.isna(td):
        return pd.NA
    total_seconds = td.total_seconds()
    minutes = int(total_seconds // 60)
    seconds = total_seconds % 60
    return f"{minutes}:{seconds:06.3f}"


def convert_gap_times_to_absolute(race_df):
    """
    Convert FastF1 race gap times (small timedeltas) to absolute times.
    Gap definition: timedelta < 10 minutes AND position != 1.
    """
    if len(race_df) == 0:
        return race_df

    race_df = race_df.copy()
    race_df["_time_td"] = race_df["time"].apply(parse_time_to_ms)
    race_df["_pos_num"] = pd.to_numeric(race_df["position"], errors="coerce")

    gap_cutoff = pd.Timedelta(minutes=10)

    # Leader absolute time per (Year, Event)
    leader_td = {}
    for (year, event), grp in race_df.groupby(["Year", "Event"]):
        leader = grp[(grp["_pos_num"] == 1) & (grp["_time_td"].notna())]
        if len(leader) > 0:
            leader_td[(year, event)] = leader.iloc[0]["_time_td"]
        else:
            # Fallback: max timedelta in group (likely absolute)
            candidates = grp["_time_td"].dropna()
            if len(candidates) > 0:
                leader_td[(year, event)] = candidates.max()

    def to_absolute_td(row):
        td = row["_time_td"]
        if pd.isna(td):
            return pd.NaT
        base = leader_td.get((row["Year"], row["Event"]))
        if base is None:
            return td
        is_gap = (td < gap_cutoff) and (row["_pos_num"] != 1)
        return base + td if is_gap else td

    race_df["_time_abs_td"] = race_df.apply(to_absolute_td, axis=1)

    # Format ALL times to match master format
    race_df["time"] = race_df["_time_abs_td"].apply(format_timedelta_to_time_str)

    race_df = race_df.drop(columns=["_time_td", "_time_abs_td", "_pos_num"], errors="ignore")
    return race_df

def categorize_status_from_statusid(statusId):
    """Categorize statusId into simplified categories (same as master dataset)."""
    if pd.isna(statusId):
        return "Unknown"
    try:
        statusId = int(statusId)
    except (ValueError, TypeError):
        return "Unknown"
    
    if statusId == 1:  # Finished
        return "Finished"
    elif statusId in [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 45, 50, 53, 55, 58, 88, 
                      111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 122, 123, 124, 125, 127, 133, 134]:
        return "Finished_Lapped"
    elif statusId == 2:  # Disqualified
        return "Disqualified"
    elif statusId == 62:  # Not classified
        return "Not_Classified"
    else:
        return "DNF"

def map_fastf1_status_to_statusid(status_text):
    """Map FastF1 Status text to statusId using status.csv lookup."""
    if pd.isna(status_text):
        return pd.NA
    status_text = str(status_text).strip()
    return status_lookup.get(status_text, pd.NA)

print("✓ Helper functions defined")

✓ Helper functions defined


## Load FastF1 Data (Year-Specific)


In [79]:
def load_fastf1_year(year):
    """
    Load FastF1 data for a specific year.
    
    Returns:
        dict: {
            'results': DataFrame (all sessions),
            'laps': DataFrame (all sessions),
            'telemetry': DataFrame (all sessions) - note: very large, use chunked reading for processing
        }
    """
    print(f"Loading FastF1 data for {year}...")
    
    results_file = FASTF1_DIR / f'ALL_RESULTS_{year}.csv'
    laps_file = FASTF1_DIR / f'ALL_LAPS_{year}.csv'
    telemetry_file = FASTF1_DIR / f'ALL_TELEMETRY_{year}.csv'
    
    data = {}
    
    # Load RESULTS
    if results_file.exists():
        results = pd.read_csv(results_file, low_memory=False)
        print(f"  ✓ RESULTS: {len(results):,} rows, {results['Session'].nunique()} sessions")
        data['results'] = results
    else:
        print(f"  ⚠ RESULTS file not found: {results_file.name}")
        data['results'] = pd.DataFrame()
    
    # Load LAPS
    if laps_file.exists():
        laps = pd.read_csv(laps_file, low_memory=False)
        print(f"  ✓ LAPS: {len(laps):,} rows, {laps['Session'].nunique()} sessions")
        data['laps'] = laps
    else:
        print(f"  ⚠ LAPS file not found: {laps_file.name}")
        data['laps'] = pd.DataFrame()
    
    # Note: Telemetry files are very large (5-6GB), we'll load them in chunks when needed
    # For now, just check if file exists
    if telemetry_file.exists():
        file_size_mb = telemetry_file.stat().st_size / (1024 * 1024)
        print(f"  ✓ TELEMETRY file exists: {file_size_mb:.1f} MB (will load in chunks when needed)")
        data['telemetry_file'] = telemetry_file
    else:
        print(f"  ⚠ TELEMETRY file not found: {telemetry_file.name}")
        data['telemetry_file'] = None
    
    return data

# Test loading 2024 data
print("Testing data loading for 2024:")
fastf1_2024 = load_fastf1_year(2024)
print()


Testing data loading for 2024:
Loading FastF1 data for 2024...
  ✓ RESULTS: 2,277 rows, 6 sessions
  ✓ LAPS: 62,690 rows, 6 sessions
  ✓ TELEMETRY file exists: 6205.5 MB (will load in chunks when needed)



## Core Race Data Extraction


In [80]:
def extract_race_data(fastf1_data, year):
    """
    Extract core race data from FastF1 RESULTS.
    
    Returns DataFrame with one row per (Year, Event, Driver) for race session.
    """
    results = fastf1_data['results']
    
    if len(results) == 0:
        return pd.DataFrame()
    
    # Filter to Race session
    race_results = results[results['Session'] == 'R'].copy()
    
    if len(race_results) == 0:
        print(f"  ⚠ No race session data found for {year}")
        return pd.DataFrame()
    
    print(f"Extracting race data from {len(race_results)} race results...")
    
    # Extract core columns
    race_df = race_results[[
        'Year', 'Event', 'Abbreviation', 'DriverNumber',
        'GridPosition', 'Position', 'Points', 'Laps', 'Time', 'Status', 'TeamName'
    ]].copy()
    
    # Rename columns to match master structure
    race_df = race_df.rename(columns={
        'Abbreviation': 'code',
        'GridPosition': 'grid',
        'Position': 'position',
        'Points': 'points',
        'Laps': 'laps',
        'Time': 'time',
        'Status': 'status_text'
    })
    
    # Handle position: convert 'R' (retired) to NaN, numeric positions to float
    def parse_position(pos):
        if pd.isna(pos):
            return pd.NA
        pos_str = str(pos).strip()
        if pos_str.upper() == 'R' or pos_str == '':
            return pd.NA
        try:
            return float(pos_str)
        except ValueError:
            return pd.NA
    
    race_df['position'] = race_df['position'].apply(parse_position)
    
    # Convert grid to numeric
    race_df['grid'] = pd.to_numeric(race_df['grid'], errors='coerce')
    
    # Convert points and laps to numeric
    race_df['points'] = pd.to_numeric(race_df['points'], errors='coerce')
    race_df['laps'] = pd.to_numeric(race_df['laps'], errors='coerce')
    
    # Map status text to statusId
    race_df['statusId'] = race_df['status_text'].apply(map_fastf1_status_to_statusid)
    
    # Calculate status_category from statusId
    race_df['status_category'] = race_df['statusId'].apply(categorize_status_from_statusid)
    
    # Convert gap times to absolute times before converting to milliseconds
    race_df = convert_gap_times_to_absolute(race_df)
    
    # Convert time to milliseconds (now all times are absolute)
    race_df['milliseconds'] = race_df['time'].apply(time_to_milliseconds)
    
    # Extract qualifying times (from Session == 'Q')
    quali_results = results[results['Session'] == 'Q'].copy()
    if len(quali_results) > 0:
        quali_df = quali_results[['Year', 'Event', 'Abbreviation', 'Q1', 'Q2', 'Q3']].copy()
        quali_df = quali_df.rename(columns={'Abbreviation': 'code'})
        
        # Merge qualifying times
        race_df = race_df.merge(
            quali_df,
            on=['Year', 'Event', 'code'],
            how='left',
            suffixes=('', '_quali')
        )
        race_df['q1'] = race_df['Q1'].apply(normalize_quali_time)
        race_df['q2'] = race_df['Q2'].apply(normalize_quali_time)
        race_df['q3'] = race_df['Q3'].apply(normalize_quali_time)
        race_df = race_df.drop(columns=['Q1', 'Q2', 'Q3'])
    else:
        race_df['q1'] = pd.NA
        race_df['q2'] = pd.NA
        race_df['q3'] = pd.NA
    
    # Add year column (already have Year, but ensure consistency)
    race_df['year'] = race_df['Year']
    
    # Standardize code to uppercase
    race_df['code'] = race_df['code'].astype(str).str.strip().str.upper()
    
    # Ensure driverIds exist for all FastF1 codes (race session)
    global drivers_df, driver_code_to_id
    drivers_df, driver_code_to_id = ensure_driver_ids(drivers_df, results)
    
    # Map driver code to driverId
    race_df['driverId'] = race_df['code'].map(driver_code_to_id)
    
    print(f"  ✓ Extracted {len(race_df)} race entries")
    print(f"  ✓ Unique events: {race_df['Event'].nunique()}")
    print(f"  ✓ Unique drivers: {race_df['code'].nunique()}")
    
    return race_df

# Test extraction for 2024
print("Testing race data extraction for 2024:")
race_data_2024 = extract_race_data(fastf1_2024, 2024)
print(f"\nSample race data:")
print(race_data_2024[['year', 'Event', 'code', 'grid', 'position', 'points', 'laps', 'statusId', 'status_category']].head(10))


Testing race data extraction for 2024:
Extracting race data from 479 race results...
  ✓ Extracted 479 race entries
  ✓ Unique events: 24
  ✓ Unique drivers: 24

Sample race data:
   year               Event code  grid  position  points  laps statusId  \
0  2024  Bahrain Grand Prix  VER   1.0       1.0    26.0  57.0        1   
1  2024  Bahrain Grand Prix  PER   5.0       2.0    18.0  57.0        1   
2  2024  Bahrain Grand Prix  SAI   4.0       3.0    15.0  57.0        1   
3  2024  Bahrain Grand Prix  LEC   2.0       4.0    12.0  57.0        1   
4  2024  Bahrain Grand Prix  RUS   3.0       5.0    10.0  57.0        1   
5  2024  Bahrain Grand Prix  NOR   7.0       6.0     8.0  57.0        1   
6  2024  Bahrain Grand Prix  HAM   9.0       7.0     6.0  57.0        1   
7  2024  Bahrain Grand Prix  PIA   8.0       8.0     4.0  57.0        1   
8  2024  Bahrain Grand Prix  ALO   6.0       9.0     2.0  57.0        1   
9  2024  Bahrain Grand Prix  STR  12.0      10.0     1.0  57.0        

In [81]:
def map_circuit_info(race_df, circuits_df, races_df):
    """
    Map circuit information (circuitId, lat, lng) and assign round numbers.
    
    Uses races.csv to get circuitId and date for each (year, name) combination,
    then maps to circuits.csv for lat/lng.
    """
    if len(race_df) == 0:
        return race_df
    
    print("Mapping circuit information...")
    
    # Load races.csv to get circuitId and date for each race
    # Create lookup: (year, name) -> (circuitId, date)
    race_lookup = {}
    for _, row in races_df.iterrows():
        key = (int(row['year']), str(row['name']).strip())
        race_lookup[key] = {
            'circuitId': int(row['circuitId']),
            'date': pd.to_datetime(row['date']),
            'round': int(row['round'])
        }
    
    # Map circuitId and date to race_df
    def get_circuit_info(row):
        key = (int(row['Year']), str(row['Event']).strip())
        return race_lookup.get(key, {})
    
    circuit_info = race_df.apply(get_circuit_info, axis=1)
    race_df['circuitId'] = [info.get('circuitId', pd.NA) for info in circuit_info]
    race_df['date'] = [info.get('date', pd.NaT) for info in circuit_info]
    race_df['round'] = [info.get('round', pd.NA) for info in circuit_info]
    
    # Map lat/lng from circuits.csv
    circuit_coords = dict(zip(circuits_df['circuitId'], zip(circuits_df['lat'], circuits_df['lng'])))
    race_df['lat'] = race_df['circuitId'].map(lambda x: circuit_coords.get(x, (pd.NA, pd.NA))[0] if pd.notna(x) else pd.NA)
    race_df['lng'] = race_df['circuitId'].map(lambda x: circuit_coords.get(x, (pd.NA, pd.NA))[1] if pd.notna(x) else pd.NA)
    
    # Add name column (same as Event)
    race_df['name'] = race_df['Event']
    
    # Check for unmapped races
    unmapped = race_df[race_df['circuitId'].isna()]
    if len(unmapped) > 0:
        print(f"  ⚠ Warning: {len(unmapped)} races could not be mapped to circuitId")
        print(f"    Unmapped events: {unmapped['Event'].unique().tolist()}")
    else:
        print(f"  ✓ All {len(race_df)} races mapped to circuits")
    
    return race_df

# Load races.csv for lookup
races_df = pd.read_csv(KAGGLE_DIR / 'races.csv', low_memory=False)
print(f"Loaded races.csv: {len(races_df)} races")
event_schedule_df = pd.read_csv(FASTF1_DIR / 'ALL_EVENT_SCHEDULE.csv', low_memory=False)

# Test circuit mapping for 2024
print("\nTesting circuit mapping for 2024:")
race_data_2024 = map_circuit_info(race_data_2024, circuits_df, races_df)
print(f"\nSample with circuit info:")
print(race_data_2024[['year', 'Event', 'circuitId', 'round', 'date', 'lat', 'lng']].head(10))


Loaded races.csv: 1125 races

Testing circuit mapping for 2024:
Mapping circuit information...
  ✓ All 479 races mapped to circuits

Sample with circuit info:
   year               Event  circuitId  round       date      lat      lng
0  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
1  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
2  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
3  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
4  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
5  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
6  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
7  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
8  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
9  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106


## Fastest Lap Calculations (from LAPS data)


In [82]:
def calculate_fastest_lap_features(race_df, laps_df):
    """
    Calculate fastestLap and fastestLapTime from ALL_LAPS data.
    
    For each (Year, Event, Driver):
    - Filter LAPS to Session == 'R', that driver
    - Find minimum LapTime (exclude 0, NaN, deleted laps)
    - fastestLap → lap number of fastest lap
    - fastestLapTime → time of fastest lap
    """
    if len(race_df) == 0 or len(laps_df) == 0:
        race_df['fastestLap'] = pd.NA
        race_df['fastestLapTime'] = pd.NA
        return race_df
    
    print("Calculating fastest lap features from LAPS data...")
    
    # Filter to race session only
    race_laps = laps_df[laps_df['Session'] == 'R'].copy()
    
    if len(race_laps) == 0:
        print("  ⚠ No race lap data found")
        race_df['fastestLap'] = pd.NA
        race_df['fastestLapTime'] = pd.NA
        return race_df
    
    # Parse lap times and filter valid laps
    def parse_lap_time(lap_time):
        """Parse lap time string to timedelta."""
        if pd.isna(lap_time):
            return pd.NaT
        td = parse_time_to_ms(lap_time)
        return td
    
    race_laps['LapTime_parsed'] = race_laps['LapTime'].apply(parse_lap_time)
    
    # Filter out invalid laps: NaN, 0, deleted laps
    valid_laps = race_laps[
        race_laps['LapTime_parsed'].notna() &
        (race_laps['LapTime_parsed'] > pd.Timedelta(0)) &
        (race_laps.get('Deleted', pd.Series([False] * len(race_laps))) != True)
    ].copy()
    
    if len(valid_laps) == 0:
        print("  ⚠ No valid lap times found")
        race_df['fastestLap'] = pd.NA
        race_df['fastestLapTime'] = pd.NA
        return race_df
    
    # Group by (Year, Event, Driver) and find fastest lap
    fastest_laps = valid_laps.loc[
        valid_laps.groupby(['Year', 'Event', 'Driver'])['LapTime_parsed'].idxmin()
    ][['Year', 'Event', 'Driver', 'LapNumber', 'LapTime', 'LapTime_parsed']].copy()
    
    fastest_laps = fastest_laps.rename(columns={
        'Driver': 'code',
        'LapNumber': 'fastestLap',
        'LapTime': 'fastestLapTime'
    })
    fastest_laps['code'] = fastest_laps['code'].astype(str).str.strip().str.upper()
    
    # Merge back to race_df
    race_df = race_df.merge(
        fastest_laps[['Year', 'Event', 'code', 'fastestLap', 'fastestLapTime']],
        on=['Year', 'Event', 'code'],
        how='left',
        suffixes=('', '_fastest')
    )
    
    race_df['fastestLapTime'] = race_df['fastestLapTime'].apply(normalize_quali_time)
    
    matched = race_df['fastestLap'].notna().sum()
    print(f"  ✓ Found fastest lap for {matched}/{len(race_df)} drivers")
    
    return race_df

# Test fastest lap calculation for 2024
print("Testing fastest lap calculation for 2024:")
race_data_2024 = calculate_fastest_lap_features(race_data_2024, fastf1_2024['laps'])
print(f"\nSample with fastest lap:")
print(race_data_2024[['year', 'Event', 'code', 'fastestLap', 'fastestLapTime']].head(10))


Testing fastest lap calculation for 2024:
Calculating fastest lap features from LAPS data...
  ✓ Found fastest lap for 465/479 drivers

Sample with fastest lap:
   year               Event code  fastestLap fastestLapTime
0  2024  Bahrain Grand Prix  VER        39.0       1:32.608
1  2024  Bahrain Grand Prix  PER        40.0       1:34.364
2  2024  Bahrain Grand Prix  SAI        44.0       1:34.507
3  2024  Bahrain Grand Prix  LEC        36.0       1:34.090
4  2024  Bahrain Grand Prix  RUS        40.0       1:35.065
5  2024  Bahrain Grand Prix  NOR        35.0       1:34.476
6  2024  Bahrain Grand Prix  HAM        39.0       1:34.722
7  2024  Bahrain Grand Prix  PIA        39.0       1:34.983
8  2024  Bahrain Grand Prix  ALO        48.0       1:34.199
9  2024  Bahrain Grand Prix  STR        30.0       1:35.632


## Fastest Lap Speed (from TELEMETRY)

**Note**: This requires matching telemetry data to specific laps using LapStartTime/LapStartDate from LAPS data. 
For now, we'll use SpeedFL from LAPS data (speed at finish line) as a proxy, or calculate from telemetry if needed.


In [83]:
def calculate_fastest_lap_speed(race_df, laps_df):
    """
    Calculate fastestLapSpeed from LAPS data (using SpeedFL - speed at finish line).
    
    For a more accurate calculation, we would need to:
    1. Match telemetry data to specific laps using LapStartTime/LapStartDate
    2. Filter telemetry to that lap's time window
    3. Calculate average Speed for that lap
    
    For now, we use SpeedFL from the fastest lap in LAPS data as a proxy.
    """
    if len(race_df) == 0 or len(laps_df) == 0:
        race_df['fastestLapSpeed'] = pd.NA
        return race_df
    
    print("Calculating fastest lap speed from LAPS data...")
    
    # Filter to race session
    race_laps = laps_df[laps_df['Session'] == 'R'].copy()
    
    if len(race_laps) == 0:
        print("  ⚠ No race lap data found")
        race_df['fastestLapSpeed'] = pd.NA
        return race_df
    
    # Get fastest lap info for each driver (already calculated in race_df)
    # Match back to LAPS to get SpeedFL for that lap
    fastest_lap_speeds = []
    
    for _, row in race_df.iterrows():
        if pd.isna(row.get('fastestLap')):
            fastest_lap_speeds.append(pd.NA)
            continue
        
        # Find the fastest lap in LAPS data
        driver_laps = race_laps[
            (race_laps['Year'] == row['Year']) &
            (race_laps['Event'] == row['Event']) &
            (race_laps['Driver'] == row['code']) &
            (race_laps['LapNumber'] == row['fastestLap'])
        ]
        
        if len(driver_laps) > 0 and 'SpeedFL' in driver_laps.columns:
            speed = driver_laps.iloc[0]['SpeedFL']
            fastest_lap_speeds.append(speed if pd.notna(speed) else pd.NA)
        else:
            fastest_lap_speeds.append(pd.NA)
    
    race_df['fastestLapSpeed'] = fastest_lap_speeds
    
    matched = race_df['fastestLapSpeed'].notna().sum()
    print(f"  ✓ Found fastest lap speed for {matched}/{len(race_df)} drivers")
    
    return race_df

# Test fastest lap speed calculation for 2024
print("Testing fastest lap speed calculation for 2024:")
race_data_2024 = calculate_fastest_lap_speed(race_data_2024, fastf1_2024['laps'])
print(f"\nSample with fastest lap speed:")
print(race_data_2024[['year', 'Event', 'code', 'fastestLap', 'fastestLapSpeed']].head(10))


Testing fastest lap speed calculation for 2024:
Calculating fastest lap speed from LAPS data...
  ✓ Found fastest lap speed for 464/479 drivers

Sample with fastest lap speed:
   year               Event code  fastestLap fastestLapSpeed
0  2024  Bahrain Grand Prix  VER        39.0           281.0
1  2024  Bahrain Grand Prix  PER        40.0           281.0
2  2024  Bahrain Grand Prix  SAI        44.0           282.0
3  2024  Bahrain Grand Prix  LEC        36.0           281.0
4  2024  Bahrain Grand Prix  RUS        40.0           282.0
5  2024  Bahrain Grand Prix  NOR        35.0           287.0
6  2024  Bahrain Grand Prix  HAM        39.0           282.0
7  2024  Bahrain Grand Prix  PIA        39.0           289.0
8  2024  Bahrain Grand Prix  ALO        48.0           280.0
9  2024  Bahrain Grand Prix  STR        30.0           281.0


## Driver Age Mapping


In [84]:
def add_driver_age(race_df, drivers_df):
    """
    Map driver age from drivers.csv using code → driverId → dob lookup.
    Calculate driver_age = race date - dob.
    """
    if len(race_df) == 0:
        race_df['driver_age'] = pd.NA
        return race_df
    
    print("Calculating driver age...")
    
    # Create driver code -> dob lookup
    driver_dob = dict(zip(
        drivers_df['code'].astype(str).str.strip().str.upper(),
        pd.to_datetime(drivers_df['dob'])
    ))
    
    # Map dob to race_df
    race_df['driver_dob'] = race_df['code'].map(driver_dob)
    
    # Calculate age at race date
    race_df['driver_age'] = (race_df['date'] - race_df['driver_dob']).dt.days / 365.25
    
    # Clean up temporary column
    race_df = race_df.drop(columns=['driver_dob'])
    
    matched = race_df['driver_age'].notna().sum()
    print(f"  ✓ Calculated age for {matched}/{len(race_df)} drivers")
    
    return race_df

# Test driver age calculation for 2024
print("Testing driver age calculation for 2024:")
race_data_2024 = add_driver_age(race_data_2024, drivers_df)
print(f"\nSample with driver age:")
print(race_data_2024[['year', 'Event', 'code', 'driverId', 'date', 'driver_age']].head(10))


Testing driver age calculation for 2024:
Calculating driver age...
  ✓ Calculated age for 479/479 drivers

Sample with driver age:
   year               Event code  driverId       date  driver_age
0  2024  Bahrain Grand Prix  VER       830 2024-03-02   26.420260
1  2024  Bahrain Grand Prix  PER       815 2024-03-02   34.097194
2  2024  Bahrain Grand Prix  SAI       832 2024-03-02   29.500342
3  2024  Bahrain Grand Prix  LEC       844 2024-03-02   26.376454
4  2024  Bahrain Grand Prix  RUS       847 2024-03-02   26.042437
5  2024  Bahrain Grand Prix  NOR       846 2024-03-02   24.301164
6  2024  Bahrain Grand Prix  HAM         1 2024-03-02   39.148528
7  2024  Bahrain Grand Prix  PIA       857 2024-03-02   22.904860
8  2024  Bahrain Grand Prix  ALO         4 2024-03-02   42.592745
9  2024  Bahrain Grand Prix  STR       840 2024-03-02   25.341547


## Sprint Results Extraction

Sprint results are attached to the same row as race data (matched by Year, Event, Driver).


In [85]:
def extract_sprint_data(race_df, fastf1_data, year):
    """
    Extract sprint results from FastF1 RESULTS and LAPS (Session == Sprint).
    Merge to same row structure as race data.
    """
    results = fastf1_data['results']
    laps = fastf1_data['laps']
    
    if len(results) == 0:
        # Add empty sprint columns
        sprint_cols = ['sprint_results_grid', 'sprint_results_positionOrder', 'sprint_results_points',
                       'sprint_results_laps', 'sprint_results_time', 'sprint_results_milliseconds',
                       'sprint_results_fastestLap', 'sprint_results_fastestLapTime', 'sprint_results_statusId']
        for col in sprint_cols:
            race_df[col] = pd.NA
        return race_df
    
    print("Extracting sprint results...")
    
    # Filter to Sprint session
    sprint_results = results[results['Session'] == 'Sprint'].copy()
    
    if len(sprint_results) == 0:
        print("  ⚠ No sprint session data found")
        sprint_cols = ['sprint_results_grid', 'sprint_results_positionOrder', 'sprint_results_points',
                       'sprint_results_laps', 'sprint_results_time', 'sprint_results_milliseconds',
                       'sprint_results_fastestLap', 'sprint_results_fastestLapTime', 'sprint_results_statusId']
        for col in sprint_cols:
            race_df[col] = pd.NA
        return race_df
    
    # Extract sprint results columns
    sprint_df = sprint_results[[
        'Year', 'Event', 'Abbreviation', 'GridPosition', 'Position', 'Points', 'Status'
    ]].copy()
    
    sprint_df = sprint_df.rename(columns={
        'Abbreviation': 'code',
        'GridPosition': 'sprint_results_grid',
        'Position': 'sprint_results_positionOrder',
        'Points': 'sprint_results_points',
        'Status': 'status_text'
    })
    sprint_df['code'] = sprint_df['code'].astype(str).str.strip().str.upper()
    
    # Map status to statusId
    sprint_df['sprint_results_statusId'] = sprint_df['status_text'].apply(map_fastf1_status_to_statusid)
    sprint_df = sprint_df.drop(columns=['status_text'])
    
    # Extract sprint lap data from LAPS
    sprint_laps = laps[laps['Session'] == 'Sprint'].copy()
    
    if len(sprint_laps) > 0:
        # Count laps per driver
        sprint_lap_counts = sprint_laps.groupby(['Year', 'Event', 'Driver']).size().reset_index(name='sprint_results_laps')
        sprint_lap_counts['Driver'] = sprint_lap_counts['Driver'].astype(str).str.strip().str.upper()
        sprint_lap_counts = sprint_lap_counts.rename(columns={'Driver': 'code'})
        
        # Calculate total time (sum of lap times)
        sprint_laps['LapTime_parsed'] = sprint_laps['LapTime'].apply(parse_time_to_ms)
        sprint_times = sprint_laps.groupby(['Year', 'Event', 'Driver'])['LapTime_parsed'].sum().reset_index()
        sprint_times['Driver'] = sprint_times['Driver'].astype(str).str.strip().str.upper()
        sprint_times = sprint_times.rename(columns={'Driver': 'code', 'LapTime_parsed': 'sprint_results_time'})
        sprint_times['sprint_results_milliseconds'] = sprint_times['sprint_results_time'].apply(
            lambda x: int(round(x.total_seconds() * 1000)) if pd.notna(x) else pd.NA
        )
        
        # Find fastest lap
        valid_sprint_laps = sprint_laps[
            sprint_laps['LapTime_parsed'].notna() &
            (sprint_laps['LapTime_parsed'] > pd.Timedelta(0))
        ]
        if len(valid_sprint_laps) > 0:
            fastest_sprint_laps = valid_sprint_laps.loc[
                valid_sprint_laps.groupby(['Year', 'Event', 'Driver'])['LapTime_parsed'].idxmin()
            ][['Year', 'Event', 'Driver', 'LapNumber', 'LapTime']].copy()
            fastest_sprint_laps['Driver'] = fastest_sprint_laps['Driver'].astype(str).str.strip().str.upper()
            fastest_sprint_laps = fastest_sprint_laps.rename(columns={
                'Driver': 'code',
                'LapNumber': 'sprint_results_fastestLap',
                'LapTime': 'sprint_results_fastestLapTime'
            })
        else:
            fastest_sprint_laps = pd.DataFrame(columns=['Year', 'Event', 'code', 'sprint_results_fastestLap', 'sprint_results_fastestLapTime'])
        
        # Merge lap data
        sprint_df = sprint_df.merge(sprint_lap_counts, on=['Year', 'Event', 'code'], how='left')
        sprint_df = sprint_df.merge(sprint_times[['Year', 'Event', 'code', 'sprint_results_time', 'sprint_results_milliseconds']], 
                                    on=['Year', 'Event', 'code'], how='left')
        sprint_df = sprint_df.merge(fastest_sprint_laps, on=['Year', 'Event', 'code'], how='left')
    else:
        sprint_df['sprint_results_laps'] = pd.NA
        sprint_df['sprint_results_time'] = pd.NA
        sprint_df['sprint_results_milliseconds'] = pd.NA
        sprint_df['sprint_results_fastestLap'] = pd.NA
        sprint_df['sprint_results_fastestLapTime'] = pd.NA
    
    # Merge sprint data to race_df
    race_df = race_df.merge(
        sprint_df,
        on=['Year', 'Event', 'code'],
        how='left',
        suffixes=('', '_sprint')
    )
    
    matched = race_df['sprint_results_grid'].notna().sum()
    print(f"  ✓ Found sprint results for {matched}/{len(race_df)} drivers")
    
    return race_df

# Test sprint extraction for 2024
print("Testing sprint results extraction for 2024:")
race_data_2024 = extract_sprint_data(race_data_2024, fastf1_2024, 2024)
print(f"\nSample with sprint results:")
sprint_cols = [col for col in race_data_2024.columns if 'sprint' in col.lower()]
if sprint_cols:
    print(race_data_2024[['year', 'Event', 'code'] + sprint_cols].head(10))
else:
    print("No sprint columns found")


Testing sprint results extraction for 2024:
Extracting sprint results...
  ✓ Found sprint results for 120/479 drivers

Sample with sprint results:
   year               Event code  sprint_results_grid  \
0  2024  Bahrain Grand Prix  VER                  NaN   
1  2024  Bahrain Grand Prix  PER                  NaN   
2  2024  Bahrain Grand Prix  SAI                  NaN   
3  2024  Bahrain Grand Prix  LEC                  NaN   
4  2024  Bahrain Grand Prix  RUS                  NaN   
5  2024  Bahrain Grand Prix  NOR                  NaN   
6  2024  Bahrain Grand Prix  HAM                  NaN   
7  2024  Bahrain Grand Prix  PIA                  NaN   
8  2024  Bahrain Grand Prix  ALO                  NaN   
9  2024  Bahrain Grand Prix  STR                  NaN   

   sprint_results_positionOrder  sprint_results_points  \
0                           NaN                    NaN   
1                           NaN                    NaN   
2                           NaN                    

In [86]:
import re

# Load constructors.csv to build TeamName -> constructorId lookup (no fallback)
constructors_df = pd.read_csv(KAGGLE_DIR / 'constructors.csv', low_memory=False)

TEAM_ALIASES = {
    # FastF1 → Kaggle constructor names
    "red bull racing": "red bull",
    "rb f1 team": "rb",
    "racing bulls": "rb",
    "alpha tauri": "alphatauri",
    "alphatauri": "alphatauri",
    "toro rosso": "toro rosso",
    "aston martin": "aston martin",
    "kick sauber": "sauber",
    "stake": "sauber",
    "sauber": "sauber",
    "alpine": "alpine f1 team",
    "haas f1 team": "haas f1 team",
    "williams": "williams",
    "mclaren": "mclaren",
    "ferrari": "ferrari",
    "mercedes": "mercedes",
}

def normalize_team_name(name):
    # Lowercase, remove quotes, collapse whitespace, strip punctuation
    s = str(name).strip().strip('"').strip("'").lower()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    # Apply alias if known
    return TEAM_ALIASES.get(s, s)

teamname_to_constructorid = dict(zip(
    constructors_df['name'].apply(normalize_team_name),
    constructors_df['constructorId']
))

print(f"Created TeamName -> constructorId lookup: {len(teamname_to_constructorid)} mappings")

def get_constructor_id(row):
    team = normalize_team_name(row.get('TeamName', ''))
    return teamname_to_constructorid.get(team, pd.NA)

Created TeamName -> constructorId lookup: 212 mappings


## Standings Calculations


In [87]:
def calculate_standings(race_df):
    """
    Calculate driver and constructor standings (points, position) from cumulative points.
    Includes PRE_RACE versions using .shift(1).
    """
    if len(race_df) == 0:
        return race_df
    
    print("Calculating standings...")
    
    # Sort by year, round, date for proper temporal ordering
    race_df = race_df.sort_values(['year', 'round', 'date']).reset_index(drop=True)
    
    # Driver standings points: cumulative sum per driver
    race_df['driver_standings_points'] = race_df.groupby('driverId')['points'].cumsum()
    
    # Driver standings position: rank by points per race
    race_df['driver_standings_position'] = (
        race_df.groupby(['year', 'round'])['driver_standings_points']
        .rank(method='min', ascending=False)
        .astype("Int64")
    )
    
    # Constructor standings points: sum of driver standings points per constructor per race
    constructor_points = race_df.groupby(['year', 'round', 'constructorId'])['driver_standings_points'].sum().reset_index()
    constructor_points = constructor_points.rename(columns={'driver_standings_points': 'constructor_standings_points'})
    race_df = race_df.merge(constructor_points, on=['year', 'round', 'constructorId'], how='left')
    
    # Constructor standings position: rank by constructor points per race
    race_df['constructor_standings_position'] = (
        race_df.groupby(['year', 'round'])['constructor_standings_points']
        .rank(method='min', ascending=False)
        .astype("Int64")
    )
    
    # PRE_RACE versions: shift by 1 grouped by driver/constructor
    race_df['driver_standings_points_PRE_RACE'] = race_df.groupby('driverId')['driver_standings_points'].shift(1)
    race_df['driver_standings_position_PRE_RACE'] = race_df.groupby('driverId')['driver_standings_position'].shift(1)
    
    # For constructor PRE_RACE, we need to recalculate from driver PRE_RACE points
    constructor_pre_points = race_df.groupby(['year', 'round', 'constructorId'])['driver_standings_points_PRE_RACE'].sum().reset_index()
    constructor_pre_points = constructor_pre_points.rename(columns={'driver_standings_points_PRE_RACE': 'constructor_standings_points_PRE_RACE'})
    race_df = race_df.merge(constructor_pre_points, on=['year', 'round', 'constructorId'], how='left', suffixes=('', '_pre'))
    
    race_df['constructor_standings_position_PRE_RACE'] = (
        race_df.groupby(['year', 'round'])['constructor_standings_points_PRE_RACE']
        .rank(method='min', ascending=False)
        .astype("Int64")
    )
    
    print(f"  ✓ Calculated driver and constructor standings")
    print(f"  ✓ Calculated PRE_RACE versions")
    
    return race_df

## Ensure constructorId exists before standings
race_data_2024['constructorId'] = race_data_2024.apply(get_constructor_id, axis=1)

print("Testing standings calculation for 2024:")
race_data_2024 = calculate_standings(race_data_2024)
print(f"\nSample with standings:")
print(race_data_2024[['year', 'Event', 'code', 'constructorId', 'points',
                      'driver_standings_points', 'driver_standings_position',
                      'constructor_standings_points', 'constructor_standings_position']].head(10))


Testing standings calculation for 2024:
Calculating standings...
  ✓ Calculated driver and constructor standings
  ✓ Calculated PRE_RACE versions

Sample with standings:
   year               Event code  constructorId  points  \
0  2024  Bahrain Grand Prix  VER              9    26.0   
1  2024  Bahrain Grand Prix  PER              9    18.0   
2  2024  Bahrain Grand Prix  SAI              6    15.0   
3  2024  Bahrain Grand Prix  LEC              6    12.0   
4  2024  Bahrain Grand Prix  RUS            131    10.0   
5  2024  Bahrain Grand Prix  NOR              1     8.0   
6  2024  Bahrain Grand Prix  HAM            131     6.0   
7  2024  Bahrain Grand Prix  PIA              1     4.0   
8  2024  Bahrain Grand Prix  ALO            117     2.0   
9  2024  Bahrain Grand Prix  STR            117     1.0   

   driver_standings_points  driver_standings_position  \
0                     26.0                          1   
1                     18.0                          2   
2        

## Additional Derived Features


In [88]:
def add_derived_features(race_df):
    """
    Add derived features: podium, status_category (already calculated), etc.
    Note: Rolling features will be calculated separately after appending to master.
    """
    if len(race_df) == 0:
        race_df['podium'] = pd.NA
        return race_df
    
    print("Adding derived features...")
    
    # Podium: 1 if position in [1,2,3], else 0
    race_df['podium'] = race_df['position'].apply(lambda x: 1 if pd.notna(x) and x in [1, 2, 3] else 0)
    
    # status_category already calculated in extract_race_data
    
    print(f"  ✓ Added podium indicator")
    print(f"  ✓ status_category already calculated")
    
    return race_df

# Test derived features for 2024
print("Testing derived features for 2024:")
race_data_2024 = add_derived_features(race_data_2024)
print(f"\nSample with derived features:")
print(race_data_2024[['year', 'Event', 'code', 'position', 'podium', 'status_category']].head(10))


Testing derived features for 2024:
Adding derived features...
  ✓ Added podium indicator
  ✓ status_category already calculated

Sample with derived features:
   year               Event code  position  podium status_category
0  2024  Bahrain Grand Prix  VER       1.0       1        Finished
1  2024  Bahrain Grand Prix  PER       2.0       1        Finished
2  2024  Bahrain Grand Prix  SAI       3.0       1        Finished
3  2024  Bahrain Grand Prix  LEC       4.0       0        Finished
4  2024  Bahrain Grand Prix  RUS       5.0       0        Finished
5  2024  Bahrain Grand Prix  NOR       6.0       0        Finished
6  2024  Bahrain Grand Prix  HAM       7.0       0        Finished
7  2024  Bahrain Grand Prix  PIA       8.0       0        Finished
8  2024  Bahrain Grand Prix  ALO       9.0       0        Finished
9  2024  Bahrain Grand Prix  STR      10.0       0        Finished


## Column Ordering and Final Structure

Reorder columns to match master_races_clean.csv structure. Select only columns that are being reproduced.


In [89]:
# Get column order from base master_races.csv (no rolling features yet)
master_columns = pd.read_csv(PROCESSED_DATA_DIR / 'master_races.csv', nrows=0).columns.tolist()

# Only keep base columns that exist in master_races.csv
reproduced_columns = master_columns.copy()

def reorder_columns(race_df, master_columns, reproduced_columns):
    """Reorder columns to match master structure and select reproduced columns."""
    # Add missing columns as NaN
    for col in reproduced_columns:
        if col not in race_df.columns:
            race_df[col] = pd.NA
    
    # Add resultId and raceId (will be set later or left as NaN)
    if 'resultId' not in race_df.columns:
        race_df['resultId'] = pd.NA
    if 'raceId' not in race_df.columns:
        # Get raceId from races.csv lookup
        race_id_lookup = dict(zip(
            zip(races_df['year'], races_df['name']),
            races_df['raceId']
        ))
        race_df['raceId'] = race_df.apply(
            lambda row: race_id_lookup.get((int(row['year']), str(row['name']).strip()), pd.NA),
            axis=1
        )
    
    # Add rank column (same as position for now)
    if 'rank' not in race_df.columns:
        race_df['rank'] = race_df['position']
    
    # Select and reorder columns
    available_cols = [col for col in reproduced_columns if col in race_df.columns]
    race_df = race_df[available_cols]
    
    # Reorder to match master column order (for columns that exist in both)
    final_order = [col for col in master_columns if col in race_df.columns]
    final_order.extend([col for col in race_df.columns if col not in final_order])
    race_df = race_df[final_order]
    
    return race_df

# Test column ordering for 2024
print("Reordering columns for 2024:")
race_data_2024 = reorder_columns(race_data_2024, master_columns, reproduced_columns)
print(f"\nFinal columns ({len(race_data_2024.columns)}):")
print(race_data_2024.columns.tolist())
print(race_data_2024.head())
print(f"\nShape: {race_data_2024.shape}")


Reordering columns for 2024:

Final columns (82):
['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId', 'year', 'round', 'circuitId', 'date', 'name', 'circuitRef', 'circuit_name', 'location', 'country', 'lat', 'lng', 'alt', 'url', 'driverRef', 'number_driver', 'code', 'forename', 'surname', 'dob', 'nationality', 'url_driver', 'constructorRef', 'name_constructor', 'nationality_constructor', 'url_constructor', 'driverStandingsId', 'driver_standings_points', 'driver_standings_position', 'driver_standings_positionText', 'wins', 'constructorStandingsId', 'constructor_standings_points', 'constructor_standings_position', 'constructor_standings_positionText', 'constructor_standings_wins', 'constructorResultsId', 'constructor_results_points', 'status', 'qualifyId', 'qualifying_constructorId', 'qualifying_number', 'qualifyin

## Complete Pipeline Function

Combine all steps into a single function for easy reproduction.


In [90]:
def reproduce_fastf1_year(year, races_df, circuits_df, drivers_df):
    """
    Complete pipeline to reproduce Kaggle-equivalent data from FastF1 for a given year.
    """
    print(f"\n{'='*80}")
    print(f"REPRODUCING FASTF1 DATA FOR {year}")
    print(f"{'='*80}\n")
    
    # Step 1: Load FastF1 data
    fastf1_data = load_fastf1_year(year)
    
    # Step 2: Extract race data
    race_df = extract_race_data(fastf1_data, year)
    if len(race_df) == 0:
        print(f"⚠ No race data found for {year}")
        return pd.DataFrame()
    
    # Step 3: Map circuit info
    race_df = map_circuit_info(race_df, circuits_df, races_df)
    
    race_df = map_event_schedule_info(race_df, event_schedule_df)
    
    # Step 4: Map constructorId (use TeamName mapping)
    race_df['constructorId'] = race_df.apply(get_constructor_id, axis=1)
    
    # Step 5: Calculate fastest lap features
    race_df = calculate_fastest_lap_features(race_df, fastf1_data['laps'])
    
    # Step 6: Calculate fastest lap speed
    race_df = calculate_fastest_lap_speed(race_df, fastf1_data['laps'])
    
    # Step 7: Add driver age
    race_df = add_driver_age(race_df, drivers_df)
    
    # Step 8: Extract sprint results
    race_df = extract_sprint_data(race_df, fastf1_data, year)
    
    # Step 9: Calculate standings
    race_df = calculate_standings(race_df)
    
    # Step 10: Add derived features
    race_df = add_derived_features(race_df)
    
    # Step 11: Reorder columns
    race_df = reorder_columns(race_df, master_columns, reproduced_columns)
    
    print(f"\n✓ Completed reproduction for {year}: {len(race_df)} rows")
    return race_df

print("✓ Complete pipeline function defined")


✓ Complete pipeline function defined


## 2024 Validation

Compare reproduced 2024 data against master_races_clean.csv to verify accuracy.

**Note**: Gap times are automatically converted to absolute race times during data extraction, so milliseconds comparison uses absolute times only.


In [91]:
# Reproduce 2024 data using complete pipeline
reproduced_2024 = reproduce_fastf1_year(2024, races_df, circuits_df, drivers_df)

# Load master 2024 data for comparison
master_2024 = pd.read_csv(PROCESSED_DATA_DIR / 'master_races_clean.csv', low_memory=False)
master_2024 = master_2024[master_2024['year'] == 2024].copy()

print(f"\n{'='*80}")
print("2024 VALIDATION")
print(f"{'='*80}\n")

print(f"Row counts:")
print(f"  Master 2024: {len(master_2024)}")
print(f"  Reproduced 2024: {len(reproduced_2024)}")
print(f"  Difference: {len(master_2024) - len(reproduced_2024)}")

# Compare key columns
comparison_cols = ['grid', 'position', 'points', 'laps', 'milliseconds', 'q1', 'q2', 'q3', 
                   'statusId', 'driver_standings_points', 'constructor_standings_points']

print(f"\nColumn comparison (matching on year, name, code):")
# Master uses 'name', reproduced also has 'name' (from map_circuit_info)
master_2024_merge = master_2024[['year', 'name', 'code'] + comparison_cols].copy()
master_2024_merge['name'] = master_2024_merge['name'].astype(str).str.strip()
master_2024_merge['code'] = master_2024_merge['code'].astype(str).str.strip().str.upper()

reproduced_2024_merge = reproduced_2024[['year', 'name', 'code'] + [c for c in comparison_cols if c in reproduced_2024.columns]].copy()
reproduced_2024_merge['name'] = reproduced_2024_merge['name'].astype(str).str.strip()
reproduced_2024_merge['code'] = reproduced_2024_merge['code'].astype(str).str.strip().str.upper()

merged = master_2024_merge.merge(
    reproduced_2024_merge,
    on=['year', 'name', 'code'],
    how='outer',
    suffixes=('_master', '_reproduced'),
    indicator=True
)

print(f"\nMerge results:")
print(f"  Both: {len(merged[merged['_merge'] == 'both'])}")
print(f"  Only master: {len(merged[merged['_merge'] == 'left_only'])}")
print(f"  Only reproduced: {len(merged[merged['_merge'] == 'right_only'])}")

# Compare values for matched rows
matched = merged[merged['_merge'] == 'both'].copy()
print(f"\nDetailed comparison for {len(matched)} matched rows:\n")

for col in comparison_cols:
    if f'{col}_master' in matched.columns and f'{col}_reproduced' in matched.columns:
        master_vals = matched[f'{col}_master'].copy()
        repro_vals = matched[f'{col}_reproduced'].copy()
        
        # Normalize \N to NA for master values
        if master_vals.dtype == 'object':
            master_vals = master_vals.replace('\\N', pd.NA)
            master_vals = master_vals.replace(r'\N', pd.NA)  # Handle escaped version
        
        # Handle position: convert both to int for comparison
        if col == 'position':
            # Convert to numeric, then to int (handles floats)
            master_vals = pd.to_numeric(master_vals, errors='coerce').astype('Int64')  # Nullable int
            repro_vals = pd.to_numeric(repro_vals, errors='coerce').astype('Int64')
        
        # Handle timing columns (q1, q2, q3): normalize format
        if col in ['q1', 'q2', 'q3']:
            def normalize_time_format(time_val):
                """Normalize time to mm:ss.mmm format."""
                if pd.isna(time_val):
                    return pd.NA
                time_str = str(time_val).strip()
                
                # Handle \N
                if time_str in ['\\N', r'\N', 'nan', 'NaT', 'None', '']:
                    return pd.NA
                
                # If already in mm:ss.mmm format, return as is
                if re.match(r'^\d+:\d{2}\.\d+$', time_str):
                    return time_str
                
                # If in timedelta format "0 days 00:01:23.821000", extract mm:ss.mmm
                if 'days' in time_str:
                    # Extract the time part after "days"
                    time_part = time_str.split('days', 1)[1].strip()
                    # Parse to get mm:ss.mmm
                    try:
                        td = pd.to_timedelta(time_part)
                        total_seconds = td.total_seconds()
                        minutes = int(total_seconds // 60)
                        seconds = total_seconds % 60
                        return f"{minutes}:{seconds:05.3f}"
                    except:
                        return pd.NA
                
                return time_str
            
            master_vals = master_vals.apply(normalize_time_format)
            repro_vals = repro_vals.apply(normalize_time_format)
        
        # Handle milliseconds: convert to numeric for comparison
        # Note: Gap times should already be converted to absolute times in extract_race_data
        if col == 'milliseconds':
            # Convert master to numeric (handles \N, strings, etc.)
            master_vals = pd.to_numeric(master_vals, errors='coerce')
            # Repro_vals should already be numeric/int, but ensure it is
            repro_vals = pd.to_numeric(repro_vals, errors='coerce')
        
        # Now do the comparison with normalized values
        both_na = master_vals.isna() & repro_vals.isna()
        master_na_only = master_vals.isna() & repro_vals.notna()
        repro_na_only = master_vals.notna() & repro_vals.isna()
        both_not_na = master_vals.notna() & repro_vals.notna()
        
        # Initialize matches array
        matches = pd.Series([False] * len(matched), index=matched.index)
        
        # Both NA = match
        matches[both_na] = True
        
        # For non-NA values, compare
        if both_not_na.sum() > 0:
            master_not_na = master_vals[both_not_na]
            repro_not_na = repro_vals[both_not_na]
            
            if col in ['q1', 'q2', 'q3']:
                # String comparison for normalized time strings
                string_matches = (master_not_na == repro_not_na)
                matches[both_not_na] = string_matches
            elif master_vals.dtype in ['float64', 'int64', 'Int64'] and repro_vals.dtype in ['float64', 'int64', 'Int64']:
                # Numeric comparison
                if master_vals.dtype == 'float64' or repro_vals.dtype == 'float64':
                    # For milliseconds, use tolerance (1 second = 1000 ms) since times may have small rounding differences
                    if col == 'milliseconds':
                        numeric_matches = abs(master_not_na - repro_not_na) < 1000
                    else:
                        numeric_matches = abs(master_not_na - repro_not_na) < 0.001
                else:
                    numeric_matches = (master_not_na == repro_not_na)
                matches[both_not_na] = numeric_matches
            else:
                # String/object comparison
                string_matches = (master_not_na == repro_not_na)
                string_matches = string_matches.fillna(False)
                matches[both_not_na] = string_matches
        
        # Calculate match rate
        match_rate = matches.sum() / len(matched) * 100
        mismatch_count = (~matches).sum()
        
        print(f"{col}:")
        print(f"  Match rate: {match_rate:.1f}% ({matches.sum()}/{len(matched)})")
        print(f"  Both NA (match): {both_na.sum()}")
        print(f"  Master NA only: {master_na_only.sum()}")
        print(f"  Reproduced NA only: {repro_na_only.sum()}")
        print(f"  Both not NA: {both_not_na.sum()}")
        
        # Show sample mismatches
        if mismatch_count > 0 and mismatch_count <= 20:
            print(f"\n  Sample mismatches ({mismatch_count} total):")
            mismatches = matched[~matches].copy()
            for idx, row in mismatches.head(10).iterrows():
                master_val = row[f'{col}_master']
                repro_val = row[f'{col}_reproduced']
                name = row.get('name', 'Unknown')
                code = row.get('code', 'Unknown')
                print(f"    {name} | {code}: Master={master_val} → Reproduced={repro_val}")
        elif mismatch_count > 20:
            print(f"\n  Sample mismatches (showing first 10 of {mismatch_count}):")
            mismatches = matched[~matches].copy()
            for idx, row in mismatches.head(10).iterrows():
                master_val = row[f'{col}_master']
                repro_val = row[f'{col}_reproduced']
                name = row.get('name', 'Unknown')
                code = row.get('code', 'Unknown')
                print(f"    {name} | {code}: Master={master_val} → Reproduced={repro_val}")
        
        print()  # Blank line between columns

print(f"\n✓ Validation complete")


REPRODUCING FASTF1 DATA FOR 2024

Loading FastF1 data for 2024...
  ✓ RESULTS: 2,277 rows, 6 sessions
  ✓ LAPS: 62,690 rows, 6 sessions
  ✓ TELEMETRY file exists: 6205.5 MB (will load in chunks when needed)
Extracting race data from 479 race results...
  ✓ Extracted 479 race entries
  ✓ Unique events: 24
  ✓ Unique drivers: 24
Mapping circuit information...
  ✓ All 479 races mapped to circuits
Calculating fastest lap features from LAPS data...
  ✓ Found fastest lap for 465/479 drivers
Calculating fastest lap speed from LAPS data...
  ✓ Found fastest lap speed for 464/479 drivers
Calculating driver age...
  ✓ Calculated age for 0/479 drivers
Extracting sprint results...
  ✓ Found sprint results for 120/479 drivers
Calculating standings...
  ✓ Calculated driver and constructor standings
  ✓ Calculated PRE_RACE versions
Adding derived features...
  ✓ Added podium indicator
  ✓ status_category already calculated

✓ Completed reproduction for 2024: 479 rows

2024 VALIDATION

Row counts:
  

## 2025 Production

Produce 2025 data using the same pipeline.


In [92]:
# Reproduce 2025 data
reproduced_2025 = reproduce_fastf1_year(2025, races_df, circuits_df, drivers_df)

# Save to CSV
output_file = PROCESSED_DATA_DIR / 'fastf1_2025_reproduced.csv'
reproduced_2025.to_csv(output_file, index=False)
print(f"\n✓ Saved 2025 data to: {output_file}")
print(f"  Shape: {reproduced_2025.shape}")
print(f"  Columns: {len(reproduced_2025.columns)}")



REPRODUCING FASTF1 DATA FOR 2025

Loading FastF1 data for 2025...
  ✓ RESULTS: 2,189 rows, 6 sessions
  ✓ LAPS: 61,402 rows, 6 sessions
  ✓ TELEMETRY file exists: 5899.1 MB (will load in chunks when needed)
Extracting race data from 459 race results...
  ✓ Extracted 459 race entries
  ✓ Unique events: 23
  ✓ Unique drivers: 21
Mapping circuit information...
  ⚠ Warning: 459 races could not be mapped to circuitId
    Unmapped events: ['Australian Grand Prix', 'Chinese Grand Prix', 'Japanese Grand Prix', 'Bahrain Grand Prix', 'Saudi Arabian Grand Prix', 'Miami Grand Prix', 'Emilia Romagna Grand Prix', 'Monaco Grand Prix', 'Spanish Grand Prix', 'Canadian Grand Prix', 'Austrian Grand Prix', 'British Grand Prix', 'Belgian Grand Prix', 'Hungarian Grand Prix', 'Dutch Grand Prix', 'Italian Grand Prix', 'Azerbaijan Grand Prix', 'Singapore Grand Prix', 'United States Grand Prix', 'Mexico City Grand Prix', 'São Paulo Grand Prix', 'Las Vegas Grand Prix', 'Qatar Grand Prix']
Calculating fastest la

## Append 2025 to Master and Save

**Note**: Rolling features will need recalculation after appending.


In [93]:
# Load base master data (no rolling features yet)
master_full = pd.read_csv(PROCESSED_DATA_DIR / 'master_races.csv', low_memory=False)

# --- Map 2025 circuitId using EventName -> name + (country, location) ---
import re
import unicodedata

def _norm(s):
    if pd.isna(s):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = s.lower()
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s

def _tokens(s):
    return set(_norm(s).split())

def _norm_country(s):
    s = _norm(s)
    if s in {"united states", "united states of america", "usa"}:
        return "usa"
    if s in {"united kingdom", "uk", "great britain"}:
        return "uk"
    if s in {"united arab emirates", "uae"}:
        return "uae"
    return s

# Load schedule (2025 only)
event_schedule_df = pd.read_csv(FASTF1_DIR / "ALL_EVENT_SCHEDULE.csv", low_memory=False)
sched_2025 = event_schedule_df[event_schedule_df["Year"] == 2025].copy()

# Build lookup: EventName -> (location, country)
event_meta = {}
for _, row in sched_2025.iterrows():
    event_meta[str(row["EventName"]).strip()] = {
        "location": row.get("Location"),
        "country": _norm_country(row.get("Country")),
    }

# Prepare circuits lookup (location + country)
circuits_lookup = circuits_df.copy()
circuits_lookup["country_norm"] = circuits_lookup["country"].apply(_norm_country)
circuits_lookup["location_tokens"] = circuits_lookup["location"].apply(_tokens)
circuits_lookup["name_tokens"] = circuits_lookup["name"].apply(_tokens)

def match_circuit_id(event_name):
    meta = event_meta.get(str(event_name).strip())
    if not meta:
        return pd.NA

    loc_tokens = _tokens(meta["location"])
    country = meta["country"]

    candidates = circuits_lookup[circuits_lookup["country_norm"] == country]
    if len(candidates) == 0:
        return pd.NA

    # 1) Try overlap against circuits.csv location
    best_id = pd.NA
    best_score = -1
    for _, c in candidates.iterrows():
        overlap = len(loc_tokens & c["location_tokens"])
        if overlap > best_score:
            best_score = overlap
            best_id = c["circuitId"]

    if best_score > 0:
        return best_id

    # 2) Fallback: overlap against circuits.csv name
    best_id = pd.NA
    best_score = -1
    for _, c in candidates.iterrows():
        overlap = len(loc_tokens & c["name_tokens"])
        if overlap > best_score:
            best_score = overlap
            best_id = c["circuitId"]

    return best_id if best_score > 0 else pd.NA

# --- Diagnostics BEFORE ---
event_col = "name"  # master column name
print("\nDIAG: CircuitId mapping")
print("Rows in reproduced_2025:", len(reproduced_2025))
print("Missing circuitId before:", reproduced_2025["circuitId"].isna().sum())
print("Unique event names in reproduced_2025:", reproduced_2025[event_col].nunique())

# Events missing from schedule
missing_events = sorted(set(reproduced_2025[event_col].unique()) - set(event_meta.keys()))
print("Events not found in ALL_EVENT_SCHEDULE:", missing_events)

# --- Apply mapping ---
reproduced_2025["circuitId"] = reproduced_2025["circuitId"].fillna(
    reproduced_2025[event_col].apply(match_circuit_id)
)

# --- Diagnostics AFTER ---
print("Missing circuitId after:", reproduced_2025["circuitId"].isna().sum())
print("Sample unmapped events:",
      reproduced_2025[reproduced_2025["circuitId"].isna()][event_col].drop_duplicates().head(10).tolist())
# --- end mapping ---

# Fill missing positionOrder using position (FastF1 reproduction leaves it blank)
if 'positionOrder' in reproduced_2025.columns:
    reproduced_2025['positionOrder'] = reproduced_2025['positionOrder'].fillna(reproduced_2025['position'])
else:
    reproduced_2025['positionOrder'] = reproduced_2025['position']

# Normalize date to YYYY-MM-DD (remove time component)
if 'date' in reproduced_2025.columns:
    reproduced_2025['date'] = pd.to_datetime(reproduced_2025['date'], errors='coerce').dt.date

# --- Assign new resultId and raceId for reproduced_2025 ---
# resultId: continues from last master row
last_result_id = pd.to_numeric(master_full['resultId'], errors='coerce').max()
last_result_id = int(last_result_id) if pd.notna(last_result_id) else 0

reproduced_2025 = reproduced_2025.copy()
reproduced_2025 = reproduced_2025.sort_values(['year', 'round', 'date', 'name', 'code']).reset_index(drop=True)

reproduced_2025['resultId'] = range(last_result_id + 1, last_result_id + 1 + len(reproduced_2025))

# raceId: increment per event (year + name)
last_race_id = pd.to_numeric(master_full['raceId'], errors='coerce').max()
last_race_id = int(last_race_id) if pd.notna(last_race_id) else 0

# Determine unique events in order
event_keys = (
    reproduced_2025[['year', 'name']]
    .drop_duplicates()
    .reset_index(drop=True)
)
event_keys['raceId'] = range(last_race_id + 1, last_race_id + 1 + len(event_keys))

# Merge raceId back into reproduced_2025
reproduced_2025 = reproduced_2025.merge(event_keys, on=['year', 'name'], how='left', suffixes=('', '_new'))
reproduced_2025['raceId'] = reproduced_2025['raceId_new']
reproduced_2025 = reproduced_2025.drop(columns=['raceId_new'])
# --- end assign ids ---

# Keep only base columns before append
reproduced_2025 = reproduced_2025[[c for c in master_full.columns if c in reproduced_2025.columns]]

# Append 2025 data
master_with_2025 = pd.concat([master_full, reproduced_2025], ignore_index=True)

# Save base+2025 (still unshifted)
output_file = PROCESSED_DATA_DIR / 'master_races_with_2025.csv'
master_with_2025.to_csv(output_file, index=False)

print(f"✓ Appended 2025 data to master")
print(f"  Original master: {len(master_full)} rows")
print(f"  With 2025: {len(master_with_2025)} rows")
print(f"  Saved to: {output_file} (base + 2025, no rolling features)")
print(f"\n⚠ NOTE: Rolling features need recalculation after append!")
print(f"  Run the rolling features calculation notebook to update:")
print(f"    - driver_points_avg_last_10")
print(f"    - driver_podium_rate_last_10")
print(f"    - dnf_rate_last_10")
print(f"    - etc.")



DIAG: CircuitId mapping
Rows in reproduced_2025: 459
Missing circuitId before: 459
Unique event names in reproduced_2025: 23
Events not found in ALL_EVENT_SCHEDULE: []
Missing circuitId after: 0
Sample unmapped events: []
✓ Appended 2025 data to master
  Original master: 12358 rows
  With 2025: 12817 rows
  Saved to: C:\Users\erikv\Downloads\F1\data\processed\master_races_with_2025.csv (base + 2025, no rolling features)

⚠ NOTE: Rolling features need recalculation after append!
  Run the rolling features calculation notebook to update:
    - driver_points_avg_last_10
    - driver_podium_rate_last_10
    - dnf_rate_last_10
    - etc.
